# E7 CAB robustness runs

Run after the full CAB/BBQ notebook finishes and after this notebook's code is committed and pushed. Use NVIDIA L4. Eight 180-prompt cells cover two models, two fairness conditions, neutral whitespace, and the frozen adversarial suffix. Each cell checkpoints every 20 prompts and resumes on the same Git commit. This subset does not yet cover validated paraphrases, identity-clause movement, stochastic seeds, or human harmful-compliance labels.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys
drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/CMPE253-Project').resolve()
assert SHARED.is_dir()
REPO = Path('/content/cmpe253-llm-demographic-bias-audit')
URL = 'https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['make', 'setup-colab'], cwd=REPO, check=True)
GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert GPU == 'NVIDIA L4', GPU
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('Ready:', GPU, COMMIT)


In [ ]:
subprocess.run(['make', 'ai-review-gate', f'ROOT={SHARED}'], cwd=REPO, check=True)
subprocess.run(['make', 'build-robustness-perturbations', f'ROOT={SHARED}'], cwd=REPO, check=True)
selection = json.loads((SHARED / 'manifests/robustness/robustness60.json').read_text())
perturb = json.loads((SHARED / 'manifests/robustness/perturbations.json').read_text())
assert len(selection['selected_families']) == 60 and perturb['variant_rows'] == 540
print('Frozen source families:', 60, 'variants:', 540)


## M1 whitespace and adversarial cells


In [ ]:
for run_id in ('ROB-SPACE-M1-B0', 'ROB-SPACE-M1-F1', 'ROB-ADV-M1-B0', 'ROB-ADV-M1-F1'):
    print('Starting', run_id, flush=True)
    subprocess.run(['make', 'run-robustness', f'ROOT={SHARED}', f'RUN_ID={run_id}'], cwd=REPO, check=True)
    receipt = json.loads((SHARED / 'manifests/robustness' / f'{run_id}.json').read_text())
    assert receipt['status'] == 'complete' and receipt['rows'] == 180 and receipt['code_commit'] == COMMIT
    print('Completed', run_id, flush=True)


## M2 whitespace and adversarial cells


In [ ]:
for run_id in ('ROB-SPACE-M2-B0', 'ROB-SPACE-M2-F1', 'ROB-ADV-M2-B0', 'ROB-ADV-M2-F1'):
    print('Starting', run_id, flush=True)
    subprocess.run(['make', 'run-robustness', f'ROOT={SHARED}', f'RUN_ID={run_id}'], cwd=REPO, check=True)
    receipt = json.loads((SHARED / 'manifests/robustness' / f'{run_id}.json').read_text())
    assert receipt['status'] == 'complete' and receipt['rows'] == 180 and receipt['code_commit'] == COMMIT
    print('Completed', run_id, flush=True)


In [ ]:
subprocess.run(['make', 'report-robustness', f'ROOT={SHARED}'], cwd=REPO, check=True)
path = SHARED / 'aggregate_results/robustness_refusal.json'
summary = json.loads(path.read_text())
assert summary['status'] == 'descriptive_automated_refusal_only' and len(summary['cells']) == 32
print('Verified automated-refusal robustness summary:', path)
